# Consultas, claves y valores

**Capítulo 5 · Universidad de las Hespérides**

Adaptación al español de *Dive into Deep Learning*, Aston Zhang, Zachary C. Lipton, Mu Li y Alexander J. Smola.
Fuente: `locked/chapter_attention-mechanisms-and-transformers/queries-keys-values.ipynb` · [Lección original](https://d2l.ai/chapter_attention-mechanisms-and-transformers/queries-keys-values.html).
Texto adaptado bajo [CC BY-SA 4.0](https://creativecommons.org/licenses/by-sa/4.0/). [Procedencia y cambios](../PROCEDENCIA.md).
Se conserva la secuencia de las celdas y de los ejercicios; las notas de Hespérides se identifican expresamente.

**Entorno:** ejecuta `uv sync` en la raíz y selecciona su Python como kernel. Las descargas se realizan una vez y quedan en `data/`.
Por defecto, el soporte limita los entrenamientos de `Trainer` a tres épocas y 1024/256 ejemplos para CPU.
Para repetir el régimen completo, inicia Jupyter con `HESPERIDES_COMPLETO=1`. Los ejemplos visuales pequeños conservan su propia configuración explícita.
Los datos de texto en inglés o francés son entradas de los experimentos originales y mantienen su idioma.


In [ ]:
from pathlib import Path
import sys
RAIZ = Path.cwd() if (Path.cwd() / "laboratorio").exists() else Path.cwd().parent
if str(RAIZ) not in sys.path:
    sys.path.insert(0, str(RAIZ))
from laboratorio import d2l, configurar, epocas
configurar()


# Consultas, claves y valores
<a id="sec_queries-keys-values"></a>

Por ejemplo, las imágenes en ImageNet son de tamaño $224 \times 224$ píxeles y CNNs están específicamente sintonizadas a este tamaño. Incluso en el procesamiento de lenguaje natural el tamaño de entrada para RNNs está bien definido y fijo. El tamaño variable se aborda mediante el procesamiento secuencial de un token a la vez, o por núcleos de convolución especialmente diseñados [Kalchbrenner.Grefenstette.Blunsom.2014](https://d2l.ai/chapter_references/zreferences.html). Este enfoque puede conducir a problemas significativos cuando la entrada es realmente de diferente tamaño con diferentes contenidos de información, como en [Referencia sec_seq2seq](https://d2l.ai/chapter_recurrent-modern/seq2seq.html#sec-seq2seq) en la transformación del texto [Sutskever.Vinyals.Le.2014](https://d2l.ai/chapter_references/zreferences.html). En particular, para secuencias largas se hace muy difícil mantener un seguimiento de todo lo que ya ha sido generado o incluso visto por la red. Incluso heurísticas de seguimiento explícitas como las propuestas por [yang2016neural](https://d2l.ai/chapter_references/zreferences.html) sólo ofrecen un beneficio limitado.

Compara esto con las bases de datos. En su forma más simple son colecciones de claves ($k$) y valores ($v$). Por ejemplo, nuestra base de datos $\mathcal{D}$ podría consistir en tuplas ("Zhang", "Aston"), ("Lipton", "Zachary"), ("Li", "Mu"), ("Smola", "Alex"), ("Hu", "Rachel"), ("Werness", "Brent"), con el último nombre como la clave y el primer nombre como el valor. Podemos operar en $\mathcal{D}$, por ejemplo con la consulta exacta ($q$) para "Li" que devolvería el valor "Mu". Si ("Li", "Mu") no fuera un registro en $\mathcal{D}$, no habría respuesta válida. Si también permitimos coincidencias aproximadas, nos recuperaríamos ("Lipton", "Zachary") en su lugar. Este ejemplo bastante simple y trivial, sin embargo, nos enseña una serie de cosas útiles:

* Podemos diseñar consultas $q$ que funcionen en pares ($k$,$v$) de tal manera que sean válidas independientemente del tamaño de la base de datos.
* La misma consulta puede recibir respuestas diferentes, según el contenido de la base de datos.
* El "código" que se ejecuta para operar en un gran espacio de estado (la base de datos) puede ser bastante simple (por ejemplo, coincidencia exacta, coincidencia aproximada, top-$k$).
* No es necesario comprimir o simplificar la base de datos para que las operaciones sean eficaces.

Claramente no habríamos introducido una base de datos simple aquí si no fuera con el propósito de explicar el aprendizaje profundo. De hecho, esto conduce a uno de los conceptos más emocionantes introducidos en el aprendizaje profundo en la última década: el *mecanismo de atención* [Bahdanau.Cho.Bengio.2014](https://d2l.ai/chapter_references/zreferences.html). Vamos a cubrir los detalles de su aplicación a la traducción automática más tarde. Por ahora, simplemente considere lo siguiente: denote por $\mathcal{D} \stackrel{\textrm{def}}{=} \{(\mathbf{k}_1, \mathbf{v}_1), \ldots (\mathbf{k}_m, \mathbf{v}_m)\}$ una base de datos de $m$ tuplas de *claves* y *valores*. Además, denote por $\mathbf{q}$ una *query*. Entonces podemos definir la *atención* sobre $\mathcal{D}$ como

$$\textrm{Attention}(\mathbf{q}, \mathcal{D}) \stackrel{\textrm{def}}{=} \sum_{i=1}^m \alpha(\mathbf{q}, \mathbf{k}_i) \mathbf{v}_i,$$

:eqlabel:`eq_attention_pooling`

donde $\alpha(\mathbf{q}, \mathbf{k}_i) \in \mathbb{R}$ ($i = 1, \ldots, m$) son pesos de atención escalar. La operación en sí misma se conoce típicamente como *atención conjunta*. El nombre *atención* deriva del hecho de que la operación presta especial atención a los términos para los que el peso $\alpha$ es significativo (es decir, grande). Como tal, la atención sobre $\mathcal{D}$ genera una combinación lineal de los valores contenidos en la base de datos. De hecho, esto contiene el ejemplo anterior como un caso especial donde todo menos un peso es cero. Tenemos un número de casos especiales:

* Los pesos $\alpha(\mathbf{q}, \mathbf{k}_i)$ no son negativos. En este caso la salida del mecanismo de atención está contenida en el cono convexo extendido por los valores $\mathbf{v}_i$.
* Los pesos $\alpha(\mathbf{q}, \mathbf{k}_i)$ forman una combinación convexa, es decir, $\sum_i \alpha(\mathbf{q}, \mathbf{k}_i) = 1$ y $\alpha(\mathbf{q}, \mathbf{k}_i) \geq 0$ para todo $i$. Este es el entorno más común en el aprendizaje profundo.
* Exactamente uno de los pesos $\alpha(\mathbf{q}, \mathbf{k}_i)$ es $1$, mientras que todos los demás son $0$. Esto es similar a una consulta de base de datos tradicional.
* Todas las ponderaciones son iguales, es decir, $\alpha(\mathbf{q}, \mathbf{k}_i) = \frac{1}{m}$ para todas las $i$. Esto equivale a un promedio en toda la base de datos, también llamada agrupación media en el aprendizaje profundo.

Una estrategia común para garantizar que los pesos suman hasta $1$ es normalizarlos a través de

$$\alpha(\mathbf{q}, \mathbf{k}_i) = \frac{\alpha(\mathbf{q}, \mathbf{k}_i)}{{\sum_j} \alpha(\mathbf{q}, \mathbf{k}_j)}.$$

En particular, para asegurar que los pesos también son no negativos, se puede recurrir a la exponenciación. Esto significa que ahora podemos elegir *cualquier* función $a(\mathbf{q}, \mathbf{k})$ y luego aplicar la operación softmax utilizado para los modelos multinomio a través de

$$\alpha(\mathbf{q}, \mathbf{k}_i) = \frac{\exp(a(\mathbf{q}, \mathbf{k}_i))}{\sum_j \exp(a(\mathbf{q}, \mathbf{k}_j))}. $$

:eqlabel:`eq_softmax_attention`

Esta operación está fácilmente disponible en todas las bibliotecas de aprendizaje profundo. Es diferenciable y su gradiente nunca desaparece, todas las cuales son propiedades deseables en un modelo. Nota, sin embargo, el mecanismo de atención introducido anteriormente no es la única opción. Por ejemplo, podemos diseñar un modelo de atención no diferenciable que se puede entrenar utilizando métodos de aprendizaje de refuerzo [Mnih.Heess.Graves.ea.2014](https://d2l.ai/chapter_references/zreferences.html). Como cabría esperar, entrenar tal modelo es bastante complejo. En consecuencia, la mayor parte de la investigación de atención moderna sigue el marco esbozado en [Referencia fig_qkv](https://d2l.ai/chapter_attention-mechanisms-and-transformers/queries-keys-values.html#fig-qkv). Así centramos nuestra exposición en esta familia de mecanismos diferenciables.

![El mecanismo de atención calcula una combinación lineal sobre valores $\mathbf{v}_\mathit{i}$ a través de la agregación de la atención,
donde los pesos se derivan de acuerdo con la compatibilidad entre una consulta $\mathbf{q}$ y las claves $\mathbf{k}_\mathit{i}$.](../recursos/originales/qkv.svg)
<a id="fig_qkv"></a>

Lo que es bastante notable es que el "código" real para ejecutar en el conjunto de claves y valores, a saber, la consulta, puede ser bastante conciso, aunque el espacio para operar encendido es significativo. Esta es una propiedad deseable para una capa de red, ya que no requiere demasiados parámetros para aprender. Igual de conveniente es el hecho de que la atención puede funcionar en bases de datos arbitrariamente grandes sin la necesidad de cambiar la forma en que se realiza la operación de agregación de la atención.


In [ ]:
import torch
from laboratorio import d2l

## Visualización
Uno de los beneficios del mecanismo de atención es que puede ser bastante intuitivo, especialmente cuando los pesos no son negativos y suma a $1$. En este caso podríamos *interpretar* pesos grandes como una manera para que el modelo seleccione componentes de relevancia. Si bien esto es una buena intuición, es importante recordar que es sólo eso, una *intuición*. Sin embargo, podemos querer visualizar su efecto en el conjunto dado de claves al aplicar una variedad de diferentes consultas. Esta función será útil más adelante.

Definimos así la función `show_heatmaps`. Tenga en cuenta que no toma una matriz (de pesos de atención) como su entrada, sino más bien un tensor con cuatro ejes, lo que permite una matriz de diferentes consultas y pesos. En consecuencia, la entrada `matrices` tiene la forma (número de filas para mostrar, número de columnas para mostrar, número de consultas, número de teclas). Esto será útil más adelante cuando queremos visualizar los trabajos que van a diseñar los transformadores.


In [ ]:
#@save
def show_heatmaps(matrices, xlabel, ylabel, titles=None, figsize=(2.5, 2.5),
                  cmap='Reds'):
    """Mostrar mapas de calor de matrices."""
    d2l.use_svg_display()
    num_rows, num_cols, _, _ = matrices.shape
    fig, axes = d2l.plt.subplots(num_rows, num_cols, figsize=figsize,
                                 sharex=True, sharey=True, squeeze=False)
    for i, (row_axes, row_matrices) in enumerate(zip(axes, matrices)):
        for j, (ax, matrix) in enumerate(zip(row_axes, row_matrices)):
            pcm = ax.imshow(matrix.detach().numpy(), cmap=cmap)
            if i == num_rows - 1:
                ax.set_xlabel(xlabel)
            if j == 0:
                ax.set_ylabel(ylabel)
            if titles:
                ax.set_title(titles[j])
    fig.colorbar(pcm, ax=axes, shrink=0.6);

### Nota docente de Hespérides

Una consulta expresa qué se busca; las claves determinan compatibilidad y los valores aportan el contenido que se mezcla. La atención produce una combinación ponderada de valores. En atención escalada, $A=\mathrm{softmax}(QK^T/\sqrt{d_k}+M)$ y $O=AV$, con softmax por fila. Una máscara causal asigna puntuación menos infinito al futuro antes de normalizar. El explorador 90 muestra estas cuatro operaciones con números pequeños y permite cambiar consulta y temperatura.

Vínculo con los apuntes: sesión 5, «Consultas, claves y valores».


Como un rápido control de la cordura vamos a visualizar la matriz de identidad, representando un caso donde el peso de la atención es $1$ sólo cuando la consulta y la clave son iguales.


In [ ]:
attention_weights = torch.eye(10).reshape((1, 1, 10, 10))
show_heatmaps(attention_weights, xlabel='Claves', ylabel='Consultas')

## Resumen
El mecanismo de atención nos permite agregar datos de muchos pares (clave, valor). Hasta ahora nuestra discusión era bastante abstracta, simplemente describiendo una forma de agrupar datos. Aún no hemos explicado de dónde podrían surgir esas misteriosas consultas, claves y valores. Algunas intuiciones podrían ayudar aquí: por ejemplo, en una configuración de regresión, la consulta podría corresponder a la ubicación donde se debe realizar la regresión. Las claves son las ubicaciones donde se observaron datos pasados y los valores son los propios valores (regresión). Este es el llamado Nadaraya--Estimador Watson [Nadaraya.1964,Watson.1964](https://d2l.ai/chapter_references/zreferences.html) que estudiaremos en la siguiente sección.

Por diseño, el mecanismo de atención proporciona un medio *diferenciable* de control mediante el cual una red neuronal puede seleccionar elementos de un conjunto y construir una suma ponderada asociada sobre representaciones.

## Ejercicios
1. Supongamos que usted quería volver a implementar aproximativas (clave, consulta) coincidencias como se utiliza en bases de datos clásicas, ¿qué función de atención elegiría?
1. Supongamos que la función de atención es dada por $a(\mathbf{q}, \mathbf{k}_i) = \mathbf{q}^\top \mathbf{k}_i$ y que $\mathbf{k}_i = \mathbf{v}_i$ para $i = 1, \ldots, m$. Denote por $p(\mathbf{k}_i; \mathbf{q})$ la distribución de probabilidad sobre las claves cuando se utiliza la normalización softmax en [Referencia eq_softmax_attention](https://d2l.ai/#eq-softmax-attention). Probar que $\nabla_{\mathbf{q}} \mathop{\textrm{Attention}}(\mathbf{q}, \mathcal{D}) = \textrm{Cov}_{p(\mathbf{k}_i; \mathbf{q})}[\mathbf{k}_i]$.
1. Diseñar un motor de búsqueda diferenciable utilizando el mecanismo de atención.
1. Revise el diseño de las redes Squeeze y Excitation [Hu.Shen.Sun.2018](https://d2l.ai/chapter_references/zreferences.html) e interpretelas a través de la lente del mecanismo de atención.


[Debate del original](https://discuss.d2l.ai/t/1592)
